获取所有 etf 的 日行情数据，需要包含这些字段。把所有的日行情数据写入到`etf_daily.csv`文件中，注意性能问题，我的内存比较小。

| 字段              | 含义       | 复权状态       | 说明                                                                     |
| --------------- | -------- | ---------- | ---------------------------------------------------------------------- |
| **`$open`**     | 开盘价      | **前复权**    | 当日开盘的前复权价格                                                             |
| **`$high`**     | 最高价      | **前复权**    | 当日最高的前复权价格                                                             |
| **`$low`**      | 最低价      | **前复权**    | 当日最低的前复权价格                                                             |
| **`$close`**    | 收盘价      | **前复权**    | 当日收盘的前复权价格                                                             |
| **`$factor`**   | 前复权因子    | —          | `factor = 前复权价 / 原始价`，用于还原真实交易价格                                       |
| **`$adjclose`** | 后复权收盘价   | **后复权**    | 以最新价为基准向后累积复权，用于计算长期真实收益率                                              |
| **`$volume`**   | 成交量      | **前复权**    | 历史成交量已按送转股比例放大，保持与当前股本口径一致                                             |
| **`$amount`**   | 成交额      | **原始/经调整** | 当日总成交金额。金额本身是真实资金，通常不复权或仅做比例调整                                         |
| **`$change`**   | 涨跌额      | —          | `close_t - close_{t-1}`，基于前复权价格计算的绝对涨跌额                                |
| **`$vwap`**     | 成交量加权平均价 | **前复权**    | 日 VWAP，近似公式为 `Σ(典型价格 × volume) / Σ(volume)`，典型价格取 `(high+low+close)/3` |


In [2]:
import pandas as pd
import time
import datetime
from jqdata import *
from IPython.display import display
import ipywidgets as widgets

# 获取所有 ETF 列表及上市/退市日期
etf_df = get_all_securities(types=['etf'])
etf_df['start_date'] = pd.to_datetime(etf_df['start_date'])
etf_df['end_date']   = pd.to_datetime(etf_df['end_date'])
etf_list = list(etf_df.index)

# 代码转换函数：000001.XSHE -> sz000001, 000001.XSHG -> sh000001
def convert_code(c):
    code, exch = c.split('.')
    return ('sz' if exch == 'XSHE' else 'sh') + code

# 建立映射
code_map = {c: convert_code(c) for c in etf_list}
start_date_map = etf_df['start_date'].to_dict()
end_date_map   = etf_df['end_date'].to_dict()

# 控制内存：小批次 + 精简字段
batch_size = 20
output_file = 'etf_daily.csv'
fields = ['open', 'high', 'low', 'close', 'factor', 'volume', 'money', 'avg']

total_batches = (len(etf_list) + batch_size - 1) // batch_size
first_batch = True
overall_start = time.time()

# 用 ipywidgets 原生进度条
progress = widgets.IntProgress(value=0, min=0, max=total_batches, description='进度:', bar_style='info')
label = widgets.Label(value=f'0/{total_batches} 批，预计剩余: 计算中...')
display(widgets.VBox([progress, label]))

for i in range(0, len(etf_list), batch_size):
    batch = etf_list[i:i + batch_size]
    batch_num = i // batch_size + 1
    batch_start = time.time()

    # 1. 获取前复权数据（不跳过停牌，用 NaN 填充停牌日）
    df = get_price(
        security=batch,
        start_date='2005-01-01',
        end_date='2026-05-18',
        frequency='daily',
        fields=fields,
        panel=False,
        skip_paused=False,
        fill_paused=False,
        fq='pre'
    )
    if df is None or df.empty:
        progress.value = batch_num
        continue

    # 重命名
    df.rename(columns={
        'open': '$open', 'high': '$high', 'low': '$low', 'close': '$close',
        'factor': '$factor', 'volume': '$volume', 'money': '$amount', 'avg': '$vwap'
    }, inplace=True)

    # 计算 change（停牌日会自动是 NaN，因为 close 是 NaN）
    df['$change'] = df.groupby('code')['$close'].diff()

    # 2. 获取后复权 close（同样不跳过停牌，NaN 填充）
    df_post = get_price(
        security=batch,
        start_date='2005-01-01',
        end_date='2026-05-18',
        frequency='daily',
        fields=['close'],
        panel=False,
        skip_paused=False,
        fill_paused=False,
        fq='post'
    )

    if df_post is not None and not df_post.empty:
        df_post.rename(columns={'close': '$adjclose'}, inplace=True)
        df = df.reset_index().merge(
            df_post.reset_index()[['time', 'code', '$adjclose']],
            on=['time', 'code'], how='left'
        ).set_index('time')
        del df_post
    else:
        df['$adjclose'] = None

    # 3. 过滤上市日期之前、退市日期之后的无效数据 + 代码转换
    df = df.reset_index()
    df['start_date'] = df['code'].map(start_date_map)
    df['end_date']   = df['code'].map(end_date_map)
    # 保留：time 在 [start_date, end_date] 闭区间内
    df = df[(df['time'] >= df['start_date']) & (df['time'] <= df['end_date'])].copy()
    df['symbol'] = df['code'].map(code_map)
    df = df.drop(columns=['code', 'start_date', 'end_date'])

    # 调整列顺序并写入
    output_cols = ['time', 'symbol', '$open', '$high', '$low', '$close', '$factor', '$adjclose',
                   '$volume', '$amount', '$change', '$vwap']
    # 确保缺失列存在（如停牌日可能出现 NaN 列缺失的情况）
    for col in output_cols:
        if col not in df.columns:
            df[col] = float('nan')
    df = df[output_cols]

    if first_batch:
        df.to_csv(output_file, index=False, encoding='utf-8-sig')
        first_batch = False
    else:
        df.to_csv(output_file, mode='a', header=False, index=False, encoding='utf-8-sig')

    del df

    # 更新进度条
    batch_elapsed = time.time() - batch_start
    remaining = total_batches - batch_num
    eta = remaining * batch_elapsed
    progress.value = batch_num
    label.value = f'{batch_num}/{total_batches} 批，本批{len(batch)}只耗时{batch_elapsed:.1f}s，预计剩余 {eta//60:.0f}分{eta%60:.0f}秒'

overall_elapsed = time.time() - overall_start
label.value = f'完成! 共{total_batches}批，总耗时 {overall_elapsed//60:.0f}分{overall_elapsed%60:.0f}秒 -> {output_file}'
progress.bar_style = 'success'

In [ ]:
import pandas as pd

# 读取已生成的 ETF 日行情数据
df = pd.read_csv('etf_daily.csv', parse_dates=['time'])

# 构造季度标签，如 2024Q1, 2024Q2
df['year'] = df['time'].dt.year
df['quarter'] = df['time'].dt.quarter
df['period'] = df['year'].astype(str) + 'Q' + df['quarter'].astype(str)

# 按 symbol + 季度 分组，计算平均日成交额
period_avg = df.groupby(['symbol', 'period'])['$amount'].mean().reset_index()
total_periods = len(period_avg)

# 筛选平均日成交额 > 2000万 的季度
period_avg_kept = period_avg[period_avg['$amount'] > 20_000_000]
kept_periods = len(period_avg_kept)
removed_periods = total_periods - kept_periods

print(f'总季度记录数: {total_periods}')
print(f'保留 (>2000万): {kept_periods} 条')
print(f'剔除 (<=2000万): {removed_periods} 条')

# 对于每个 symbol+period，获取该季度实际的起止日期
# 先合并回去过滤后的 period
df_filtered = df.merge(period_avg_kept[['symbol', 'period']], on=['symbol', 'period'], how='inner')

# 按 symbol + period 聚合起止日期
summary = df_filtered.groupby(['symbol', 'period'])['time'].agg(['min', 'max']).reset_index()
summary.columns = ['code', 'period', 'start_date', 'end_date']

# 格式化日期
summary['start_date'] = summary['start_date'].dt.strftime('%Y-%m-%d')
summary['end_date'] = summary['end_date'].dt.strftime('%Y-%m-%d')

# 删除 period 列，保留 3 列输出
summary = summary[['code', 'start_date', 'end_date']]

# 保存结果
summary.to_csv('etf_date_range.csv', index=False, encoding='utf-8-sig')
print(f'\n共输出 {len(summary)} 条记录（符合条件的 symbol+季度）')
print(summary.head(10))